# 08 - Module 4: LLM-agent decision evaluation

Three frontier LLMs (Claude Opus 4.7, Gemini 3.1 Pro, GPT 5.4) judge a
set of ambiguous transactions under three information conditions
(C1/C2/C3). The unified access is done via OpenRouter; configure your
key in `.env`.

WARNING: this notebook issues around 2700 API calls at the default
settings (100 transactions x 3 conditions x 3 agents x 3 configs). Expect
a cost on the order of 10-20 USD and a runtime of 2-4 hours.


In [ ]:
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from xai_blockchain_framework.config import AGENT_DISPLAY_NAMES, CONFIG, PATHS, set_seed
from xai_blockchain_framework.llm import (
    OpenRouterClient, SYSTEM_PROMPT, build_prompts, call_agent, parse_response,
)
from xai_blockchain_framework.metrics.llm import (
    cohen_kappa_pair, decision_accuracy,
    expected_calibration_error, explanation_utilization,
)
from xai_blockchain_framework.models import load_ml_model
from xai_blockchain_framework.utils import save_csv

set_seed()
assert CONFIG.has_llm_key(), 'Set OPENROUTER_API_KEY in .env first.'
print('Models in use:')
for k, v in CONFIG.models.items():
    print(f'  {k}: {v}')


## Select ambiguous transactions and load attributions

In [ ]:
N_TX = 100
AMBIG_LOW, AMBIG_HIGH = 0.3, 0.7
TOP_K = 5

splits = {
    'Elliptic': np.load(PATHS.data_processed / 'elliptic_splits.npz'),
    'Ethereum': np.load(PATHS.data_processed / 'ethereum_splits.npz'),
}
indices = {
    'Elliptic': np.load(PATHS.results_dir / 'xai_eval_indices_elliptic.npy'),
    'Ethereum': np.load(PATHS.results_dir / 'xai_eval_indices_ethereum.npy'),
}

m1 = pd.read_csv(PATHS.results_dir / 'module1_fidelity_results.csv')
m1 = m1[m1['k'] == 5]
m2 = pd.read_csv(PATHS.results_dir / 'module2_stability_results.csv')
m3 = pd.read_csv(PATHS.results_dir / 'module3_bras_results.csv')


def select_configs_by_bras(m3_df: pd.DataFrame, dataset: str) -> list[dict]:
    """Return the highest-BRAS and lowest-BRAS configurations for a dataset.

    Mirrors the original research recipe that contrasts a 'good' XAI tier
    against a 'bad' one. When only one config exists it is returned alone.
    """
    sub = m3_df[m3_df['Dataset'] == dataset].copy()
    if sub.empty or 'BRAS' not in sub.columns:
        return []
    best = sub.loc[sub['BRAS'].idxmax()]
    worst = sub.loc[sub['BRAS'].idxmin()]
    configs = [{
        'dataset': dataset,
        'model': best['Model'].lower(),
        'xai': best['XAI'].lower(),
        'bras_tier': 'high',
        'bras_label': f"high_BRAS={float(best['BRAS']):.3f}",
        'bras_value': float(best['BRAS']),
    }]
    if float(worst['BRAS']) != float(best['BRAS']):
        configs.append({
            'dataset': dataset,
            'model': worst['Model'].lower(),
            'xai': worst['XAI'].lower(),
            'bras_tier': 'low',
            'bras_label': f"low_BRAS={float(worst['BRAS']):.3f}",
            'bras_value': float(worst['BRAS']),
        })
    return configs


configs = select_configs_by_bras(m3, 'Elliptic') + select_configs_by_bras(m3, 'Ethereum')
print('Selected configs:')
for c in configs:
    print(f"  {c['bras_label']:<20} {c['dataset']:<9} {c['model']:<13} {c['xai']}")


## Call the three agents on each transaction

In [ ]:
# Main call loop with incremental checkpointing. Uses the original
# progressive-fallback ambiguous selection (strict -> [0.2,0.8] ->
# [0.1,0.9] -> closest-to-0.5) so that every configuration reliably
# yields N_TX transactions, balanced between fraud and legitimate.
rng = np.random.default_rng(CONFIG.random_seed)
client = OpenRouterClient()

CSV_PATH = PATHS.results_dir / 'module4_llm_raw_responses.csv'

completed_keys = set()
if CSV_PATH.exists():
    existing = pd.read_csv(CSV_PATH)
    for _, r in existing.iterrows():
        completed_keys.add((str(r['BRAS_label']), int(r['TX_idx']),
                            str(r['Condition']), str(r['Agent'])))
    print(f'Resuming: {len(completed_keys)} responses already in CSV.')

CSV_COLUMNS = [
    'Dataset', 'BRAS_label', 'BRAS_tier', 'Model', 'XAI', 'Condition', 'Agent',
    'TX_idx', 'Ground_Truth', 'P_fraud',
    'Decision', 'Confidence', 'Reasoning', 'Explanation', 'Error',
]


def append_row(row: dict) -> None:
    df = pd.DataFrame([{k: row.get(k, '') for k in CSV_COLUMNS}])
    header = not CSV_PATH.exists()
    df.to_csv(CSV_PATH, mode='a', header=header, index=False)


def select_ambiguous_transactions(y_test, eval_idx, model, X_test, n=N_TX, seed=42):
    """Progressive fallback ambiguous sampling from the original paper.

    Tries, in order: [AMBIG_LOW, AMBIG_HIGH], [0.2, 0.8], [0.1, 0.9], and
    finally the n transactions closest to 0.5. Then balances fraud and
    legitimate classes as much as possible.
    """
    np.random.seed(seed)
    probs = model.predict_proba(X_test[eval_idx])[:, 1]
    for lo, hi in [(AMBIG_LOW, AMBIG_HIGH), (0.2, 0.8), (0.1, 0.9)]:
        mask = (probs >= lo) & (probs <= hi)
        if mask.sum() >= n:
            ambig = eval_idx[mask]
            break
    else:
        ambiguity = np.abs(probs - 0.5)
        sorted_idx = np.argsort(ambiguity)
        ambig = eval_idx[sorted_idx[:max(n, 2 * n)]]

    fraud = ambig[y_test[ambig] == 1]
    legit = ambig[y_test[ambig] == 0]
    nf = min(n // 2, len(fraud))
    nl = min(n - nf, len(legit))
    if nf + nl < n:
        nf = min(n - nl, len(fraud))
    sel_f = np.random.choice(fraud, nf, replace=False) if nf > 0 else np.array([], dtype=int)
    sel_l = np.random.choice(legit, nl, replace=False) if nl > 0 else np.array([], dtype=int)
    out = np.concatenate([sel_f, sel_l])
    np.random.shuffle(out)
    return out


for cfg in configs:
    ds = cfg['dataset']; model_key = cfg['model']; xai = cfg['xai']
    tier = cfg['bras_tier']; bras_label = cfg['bras_label']
    sp = splits[ds]; eval_idx = indices[ds]
    X_test, y_test = sp['X_test'], sp['y_test']

    model = load_ml_model(PATHS.models_dir / f'{ds.lower()}_{model_key}.joblib')
    tx_indices = select_ambiguous_transactions(y_test, eval_idx, model, X_test)
    probs_full = model.predict_proba(X_test)[:, 1]

    attrs_full = np.load(PATHS.results_dir / f'{xai}_{model_key}_{ds.lower()}.npy')
    # Map each selected tx_idx back to its position inside eval_idx so we
    # can look up the corresponding attribution vector.
    eval_positions = {int(v): i for i, v in enumerate(eval_idx)}

    n_fraud = int((y_test[tx_indices] == 1).sum())
    n_legit = len(tx_indices) - n_fraud
    print(f'[{ds}] {bras_label} ({model_key}-{xai}): {len(tx_indices)} tx  (fraud={n_fraud}, legit={n_legit})')

    q1 = m1[(m1['Model'] == model_key.upper()) & (m1['XAI'] == xai.upper()) & (m1['Dataset'] == ds)]
    q2 = m2[(m2['Model'] == model_key.upper()) & (m2['XAI'] == xai.upper()) & (m2['Dataset'] == ds)]
    q3 = m3[(m3['Model'] == model_key.upper()) & (m3['XAI'] == xai.upper()) & (m3['Dataset'] == ds)]
    quality: dict[str, float] = {}
    for col in ['Comprehensiveness', 'Sufficiency', 'Infidelity']:
        if not q1.empty:
            quality[col] = float(q1.iloc[0][col])
    for col in ['Kendall_tau', 'Identity', 'Lipschitz_norm']:
        if not q2.empty and col in q2.columns:
            quality[col] = float(q2.iloc[0][col])
    for col in ['RAS', 'DVR', 'BRAS']:
        if not q3.empty:
            quality[col] = float(q3.iloc[0][col])

    for tx_idx in tqdm(tx_indices, desc=f'{ds} {bras_label}'):
        x = X_test[tx_idx]
        gt = int(y_test[tx_idx])
        p = float(probs_full[tx_idx])
        pos = eval_positions.get(int(tx_idx))
        if pos is None or pos >= len(attrs_full):
            attr_vec = np.zeros(x.shape[0])
        else:
            attr_vec = attrs_full[pos]
        prompts = build_prompts(
            dataset=ds, x=x,
            model_name=model_key.upper(),
            model_fraud_probability=p,
            attributions=attr_vec,
            feature_names=None,
            quality_scores=quality,
            top_k=TOP_K,
        )
        for condition in ['C1', 'C2', 'C3']:
            for agent_id in ['opus', 'gemini', 'gpt']:
                agent_display = AGENT_DISPLAY_NAMES[agent_id]
                key = (bras_label, int(tx_idx), condition, agent_display)
                if key in completed_keys:
                    continue
                try:
                    resp = call_agent(agent_id, SYSTEM_PROMPT,
                                      prompts[condition], client=client)
                    parsed = parse_response(resp.content)
                    err_msg = ''
                except Exception as err:
                    parsed = None
                    err_msg = str(err)
                row = {
                    'Dataset': ds, 'BRAS_label': bras_label, 'BRAS_tier': tier,
                    'Model': model_key.upper(), 'XAI': xai.upper(),
                    'Condition': condition, 'Agent': agent_display,
                    'TX_idx': int(tx_idx),
                    'Ground_Truth': 'fraud' if gt == 1 else 'legitimate',
                    'P_fraud': p,
                    'Decision': parsed.decision if parsed else None,
                    'Confidence': parsed.confidence if parsed else None,
                    'Reasoning': parsed.reasoning if parsed else '',
                    'Explanation': parsed.explanation if parsed else '',
                    'Error': err_msg,
                }
                append_row(row)
                completed_keys.add(key)

raw_df = pd.read_csv(CSV_PATH)
print(f'Total responses in CSV: {len(raw_df)}')


## Aggregate metrics per (dataset, condition, agent)

In [ ]:
from itertools import combinations
from sklearn.metrics import cohen_kappa_score

valid = raw_df.dropna(subset=['Decision']).copy()
valid['Correct'] = (valid['Decision'] == valid['Ground_Truth']).astype(int)

summary_rows = []
group_cols = ['Dataset', 'BRAS_label', 'BRAS_tier', 'Condition']
for keys, grp in valid.groupby(group_cols):
    ds, bras_label, bras_tier, cond = keys
    agents = list(grp['Agent'].unique())
    per_agent = {}
    for agent, a_grp in grp.groupby('Agent'):
        per_agent[agent] = {
            'DA': decision_accuracy(a_grp['Decision'], (a_grp['Ground_Truth'] == 'fraud').astype(int)),
            'ECE': expected_calibration_error(a_grp['Confidence'], a_grp['Correct']),
            'N': len(a_grp),
        }
    pivot = grp.pivot_table(index='TX_idx', columns='Agent', values='Decision', aggfunc='first')
    pivot = pivot.dropna()
    kappas = []
    if len(pivot) >= 2 and len(agents) >= 2:
        for a, b in combinations(agents, 2):
            if a not in pivot.columns or b not in pivot.columns:
                continue
            da = (pivot[a] == 'fraud').astype(int).tolist()
            db = (pivot[b] == 'fraud').astype(int).tolist()
            if len(set(da)) < 2 and len(set(db)) < 2:
                continue
            kappas.append(cohen_kappa_score(da, db))
    mean_kappa = float(np.mean(kappas)) if kappas else float('nan')
    da_values = [v['DA'] for v in per_agent.values()]
    ece_values = [v['ECE'] for v in per_agent.values()]
    summary_rows.append({
        'Dataset': ds, 'BRAS_label': bras_label, 'BRAS_tier': bras_tier,
        'Condition': cond,
        'N_tx': len(pivot),
        'DA_mean': float(np.mean(da_values)) if da_values else float('nan'),
        'DA_std': float(np.std(da_values)) if da_values else float('nan'),
        'ECE_mean': float(np.mean(ece_values)) if ece_values else float('nan'),
        'Kappa_interagent': mean_kappa,
    })

summary = pd.DataFrame(summary_rows).sort_values(['Dataset', 'BRAS_tier', 'Condition'])
print(summary.to_string(index=False))
save_csv(summary, PATHS.results_dir / 'module4_llm_summary.csv')
